# 07. prev2 거래 feature 실험

E06 reference 대비 `prev2` 가격/공백/trend feature가 tail risk, 장기 거래 공백, recent_holdout 안정성을 개선하는지 `F09`, `F10` 두 후보만 노출해 검증합니다.

이 노트북은 `transactions.csv`, `scripts/build_transactions_dataset.py`, 운영 API/DB schema를 수정하지 않습니다. 새 산출물은 모두 `outputs/e07_prev2_*` prefix로만 생성합니다.


In [ ]:

from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from datetime import date
from pathlib import Path
import csv
import importlib.util
import json
import math
import os
import random
import sys
import time
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from IPython.display import display
except Exception:  # plain-python runner fallback
    def display(value):
        print(value)

warnings.filterwarnings("ignore", category=FutureWarning)
print("python", sys.version)
print("tensorflow", tf.__version__)
print("pandas", pd.__version__)


In [ ]:

# 1) 경로와 실행 설정
current_dir = Path.cwd()
if current_dir.name == "final_project":
    PROJECT_DIR = current_dir
elif (current_dir / "final_project").exists():
    PROJECT_DIR = current_dir / "final_project"
else:
    PROJECT_DIR = Path("/Users/gwongwangjae/goorm-ai-language-course/final_project")

DATA_PATH = PROJECT_DIR / "data" / "processed" / "transactions.csv"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
SCRIPT_PATH = PROJECT_DIR / "scripts" / "build_transactions_dataset.py"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PATH = OUTPUT_DIR / "e07_prev2_features.csv"
FEATURE_QUALITY_REPORT_PATH = OUTPUT_DIR / "e07_prev2_feature_quality_report.md"
METRICS_PATH = OUTPUT_DIR / "e07_prev2_metrics.csv"
GROUP_METRICS_PATH = OUTPUT_DIR / "e07_prev2_group_metrics.csv"
SUMMARY_PATH = OUTPUT_DIR / "e07_prev2_summary.md"

RUN_MODE = os.environ.get("E07_RUN_MODE", "full").strip().lower()
REBUILD_FEATURES = os.environ.get("E07_REBUILD_FEATURES", "0") == "1"
RANDOM_STATE = 42
SMOKE_LIMITS = {"train": 200_000, "valid": 50_000, "test": 50_000, "recent_holdout": 50_000}
SPLIT_ORDER = ["train", "valid", "test", "recent_holdout"]
EVAL_SPLITS = ["valid", "test", "recent_holdout"]

BATCH_SIZE = int(os.environ.get("E07_BATCH_SIZE", "8192"))
MAX_EPOCHS = int(os.environ.get("E07_MAX_EPOCHS", "30"))
EARLY_STOPPING_PATIENCE = int(os.environ.get("E07_EARLY_STOPPING_PATIENCE", "4"))
MIN_GROUP_ROWS_FOR_SUMMARY = 100
ERROR_RATE_THRESHOLDS = [0.10, 0.20, 0.30, 0.50]

REFERENCE_E06_LOG_MAE = {"valid": 0.065917, "test": 0.067010, "recent_holdout": 0.072772}

assert RUN_MODE in {"smoke", "full"}, RUN_MODE
assert DATA_PATH.exists(), DATA_PATH
assert INTERIM_DIR.exists(), INTERIM_DIR
assert SCRIPT_PATH.exists(), SCRIPT_PATH
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
print("project", PROJECT_DIR)
print("run_mode", RUN_MODE, "max_epochs", MAX_EPOCHS, "batch_size", BATCH_SIZE)


In [ ]:

# 2) prev2 sidecar 생성 helper
spec = importlib.util.spec_from_file_location("transactions_dataset_builder", SCRIPT_PATH)
builder = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = builder
assert spec.loader is not None
spec.loader.exec_module(builder)

RAW_PREV2_COLUMNS = [
    "transaction_id", "complex_prev2_price_per_m2", "prev2_missing", "prev2_gap_days", "prev2_source_deal_date"
]
SIDECAR_COLUMNS = [
    "transaction_id", "complex_prev2_price_per_m2", "prev2_missing", "prev2_gap_days",
    "log_complex_prev2_price_per_m2", "prev1_prev2_log_return", "prev1_prev2_gap_days",
    "prev2_source_deal_date",
]

@dataclass(frozen=True)
class PrevHist:
    area: float
    deal_date: date
    ppm: float
    seq: int


def _to_float(value):
    if value is None:
        return None
    try:
        out = float(value)
    except Exception:
        return None
    return out if math.isfinite(out) else None


def add_hist_prev2(hist, row, seq):
    if row["is_cancelled"] or not row["complex_id"] or not row["deal_date_obj"]:
        return seq
    area = _to_float(row["area_m2"])
    ppm = _to_float(row["price_per_m2"])
    if area is None or area <= 0 or ppm is None or ppm <= 0:
        return seq
    seq += 1
    hist[row["complex_id"]][math.floor(area)].append(PrevHist(area, row["deal_date_obj"], ppm, seq))
    return seq


def find_prev_two(hist, complex_id, area_value, deal_dt):
    area = _to_float(area_value)
    if area is None or area <= 0 or not complex_id or not deal_dt:
        return []
    buckets = hist.get(complex_id)
    if not buckets:
        return []
    lo, hi = area * 0.9, area * 1.1
    candidates = []
    for bucket in range(math.floor(lo), math.floor(hi) + 1):
        items = buckets.get(bucket)
        if not items:
            continue
        found = 0
        for item in reversed(items):
            if item.deal_date >= deal_dt:
                continue
            if lo <= item.area <= hi:
                candidates.append(item)
                found += 1
                if found >= 2:
                    break
    candidates.sort(key=lambda item: (item.deal_date, item.seq), reverse=True)
    return candidates[:2]


def write_raw_prev2_sidecar(raw_tmp_path):
    paths = sorted(INTERIM_DIR.glob("transactions_base_*.csv.gz"))
    assert paths, INTERIM_DIR
    hist = defaultdict(lambda: defaultdict(list))
    seen = set()
    current_day = None
    day_records = []
    seq = 0
    stats = {
        "base_rows": 0,
        "sidecar_rows": 0,
        "prev2_missing_rows": 0,
        "collision_rows": 0,
        "source_date_failures": 0,
    }

    def flush(records, writer):
        nonlocal seq
        if not records:
            return
        for row in records:
            prevs = find_prev_two(hist, row["complex_id"], row["area_m2"], row["deal_date_obj"])
            prev2 = prevs[1] if len(prevs) >= 2 else None
            if not row["deal_date_obj"] or row["deal_date_obj"] < date(2019, 1, 1):
                continue
            reason = builder.exclusion(row)
            if reason:
                continue
            tid = row["transaction_id"]
            if tid in seen:
                tid = f"{tid}_{row['trade_id']}"
                row["transaction_id"] = tid
                stats["collision_rows"] += 1
            if tid in seen:
                raise RuntimeError(f"duplicate transaction_id after collision handling: {tid}")
            seen.add(tid)
            if prev2 is None:
                stats["prev2_missing_rows"] += 1
                writer.writerow({
                    "transaction_id": tid,
                    "complex_prev2_price_per_m2": "",
                    "prev2_missing": "1",
                    "prev2_gap_days": "",
                    "prev2_source_deal_date": "",
                })
            else:
                gap_days = (row["deal_date_obj"] - prev2.deal_date).days
                if gap_days <= 0:
                    stats["source_date_failures"] += 1
                writer.writerow({
                    "transaction_id": tid,
                    "complex_prev2_price_per_m2": f"{prev2.ppm:.12f}",
                    "prev2_missing": "0",
                    "prev2_gap_days": str(gap_days),
                    "prev2_source_deal_date": prev2.deal_date.isoformat(),
                })
            stats["sidecar_rows"] += 1
        for row in records:
            seq = add_hist_prev2(hist, row, seq)

    with raw_tmp_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RAW_PREV2_COLUMNS)
        writer.writeheader()
        for raw_row in builder.iter_rows(paths):
            stats["base_rows"] += 1
            row = builder.prepare(raw_row)
            deal_dt = row["deal_date_obj"]
            if current_day is None:
                current_day = deal_dt
            if deal_dt != current_day:
                flush(day_records, writer)
                day_records = []
                current_day = deal_dt
            day_records.append(row)
        flush(day_records, writer)
    return stats


def md_table(frame, floatfmt=".6f"):
    x = frame.copy()
    for col in x.select_dtypes(include=["float", "float32", "float64"]).columns:
        x[col] = x[col].map(lambda v: format(v, floatfmt) if pd.notna(v) else "")
    x = x.astype("string").fillna("")
    lines = ["| " + " | ".join(x.columns) + " |", "| " + " | ".join(["---"] * len(x.columns)) + " |"]
    lines += ["| " + " | ".join(map(str, row)) + " |" for row in x.values.tolist()]
    return "\n".join(lines)


In [ ]:

# 3) transactions.csv 로드, sidecar 보장, feature 품질 리포트
BASE_USECOLS = [
    "transaction_id", "complex_id", "legal_dong_code", "sgg_code", "area_m2", "floor", "age_years",
    "deal_date", "trade_type", "is_cancelled", "price_total", "price_per_m2",
    "complex_prev_price_per_m2", "complex_prev_missing", "prev_deal_gap_days",
]
BASE_DTYPES = {
    "transaction_id": "string", "complex_id": "string", "legal_dong_code": "string", "sgg_code": "string",
    "area_m2": "float32", "floor": "float32", "age_years": "float32", "trade_type": "string",
    "is_cancelled": "Int8", "price_total": "float32", "price_per_m2": "float32",
    "complex_prev_price_per_m2": "float32", "complex_prev_missing": "Int8", "prev_deal_gap_days": "float32",
}
raw_df = pd.read_csv(DATA_PATH, usecols=BASE_USECOLS, dtype=BASE_DTYPES, parse_dates=["deal_date"])
print("transactions rows", len(raw_df))


def build_prev2_feature_file(transactions_df):
    raw_tmp = OUTPUT_DIR / "e07_prev2_features.raw.tmp.csv"
    if raw_tmp.exists():
        raw_tmp.unlink()
    start = time.perf_counter()
    raw_stats = write_raw_prev2_sidecar(raw_tmp)
    print("raw prev2 sidecar", raw_stats, "seconds", round(time.perf_counter() - start, 2))
    prev2_raw = pd.read_csv(
        raw_tmp,
        dtype={"transaction_id": "string", "prev2_missing": "Int8"},
        parse_dates=["prev2_source_deal_date"],
    )
    assert prev2_raw["transaction_id"].is_unique
    base = transactions_df[[
        "transaction_id", "deal_date", "complex_prev_price_per_m2", "complex_prev_missing", "prev_deal_gap_days"
    ]].copy()
    merged = base.merge(prev2_raw, on="transaction_id", how="left", validate="one_to_one")
    join_missing = int(merged["prev2_missing"].isna().sum())
    assert join_missing == 0, join_missing

    prev2_price = pd.to_numeric(merged["complex_prev2_price_per_m2"], errors="coerce").astype("float64")
    prev1_price = pd.to_numeric(merged["complex_prev_price_per_m2"], errors="coerce").astype("float64")
    prev2_gap = pd.to_numeric(merged["prev2_gap_days"], errors="coerce").astype("float64")
    prev1_gap = pd.to_numeric(merged["prev_deal_gap_days"], errors="coerce").astype("float64")

    merged["complex_prev2_price_per_m2"] = prev2_price
    merged["prev2_gap_days"] = prev2_gap
    merged["prev2_missing"] = merged["prev2_missing"].fillna(1).astype("Int8")
    merged["log_complex_prev2_price_per_m2"] = np.where(prev2_price > 0, np.log(prev2_price), np.nan)
    merged["prev1_prev2_log_return"] = np.where((prev1_price > 0) & (prev2_price > 0), np.log(prev1_price) - np.log(prev2_price), np.nan)
    merged["prev1_prev2_gap_days"] = np.where(prev2_gap.notna() & prev1_gap.notna(), prev2_gap - prev1_gap, np.nan)

    sidecar = merged[SIDECAR_COLUMNS].copy()
    sidecar.to_csv(FEATURE_PATH, index=False)
    raw_tmp.unlink(missing_ok=True)
    return raw_stats


def validate_prev2_sidecar(transactions_df, sidecar_df, raw_stats=None):
    base = transactions_df[[
        "transaction_id", "deal_date", "complex_prev_price_per_m2", "complex_prev_missing", "prev_deal_gap_days"
    ]].copy()
    merged = base.merge(sidecar_df, on="transaction_id", how="left", validate="one_to_one")
    join_missing = int(merged["prev2_missing"].isna().sum())
    missing = merged["prev2_missing"].astype("Int8").eq(1)
    present = merged["prev2_missing"].astype("Int8").eq(0)
    source_dates = pd.to_datetime(merged["prev2_source_deal_date"], errors="coerce")
    deal_dates = pd.to_datetime(merged["deal_date"], errors="coerce")
    prev2_price = pd.to_numeric(merged["complex_prev2_price_per_m2"], errors="coerce")
    prev2_gap = pd.to_numeric(merged["prev2_gap_days"], errors="coerce")
    log_prev2 = pd.to_numeric(merged["log_complex_prev2_price_per_m2"], errors="coerce")
    log_return = pd.to_numeric(merged["prev1_prev2_log_return"], errors="coerce")
    prev12_gap = pd.to_numeric(merged["prev1_prev2_gap_days"], errors="coerce")
    prev1_missing = merged["complex_prev_missing"].astype("Int8").eq(1)

    expected_gap = (deal_dates - source_dates).dt.days.astype("float64")
    checks = {
        "row_count_match": len(sidecar_df) == len(transactions_df),
        "transaction_id_unique": sidecar_df["transaction_id"].is_unique,
        "join_missing_zero": join_missing == 0,
        "prev2_missing_null_fields": bool((prev2_price[missing].isna() & prev2_gap[missing].isna() & source_dates[missing].isna()).all()),
        "prev2_present_fields": bool((prev2_price[present].gt(0) & prev2_gap[present].gt(0) & source_dates[present].notna()).all()),
        "prev2_source_before_deal": bool((source_dates[present] < deal_dates[present]).all()),
        "prev2_gap_consistent": bool((prev2_gap[present].round(0).to_numpy() == expected_gap[present].round(0).to_numpy()).all()),
        "prev2_log_finite": bool(np.isfinite(log_prev2[present]).all()),
        "prev1_prev2_log_return_consistent": bool(log_return[present & ~prev1_missing].notna().all()),
        "prev1_prev2_gap_non_negative": bool(prev12_gap[present & ~prev1_missing].ge(0).all()),
    }
    grade = "Pass" if all(checks.values()) else "Fail"
    quantiles = prev2_gap[present].quantile([0.5, 0.75, 0.90, 0.95, 0.99]).to_dict() if present.any() else {}
    yearly = merged.assign(year=deal_dates.dt.year, present=present).groupby("year", dropna=False).agg(
        rows=("transaction_id", "size"), prev2_present=("present", "sum")
    ).reset_index()
    yearly["prev2_missing"] = yearly["rows"] - yearly["prev2_present"]
    yearly["prev2_missing_rate"] = yearly["prev2_missing"] / yearly["rows"]

    lines = []
    lines.append("# E07 prev2 feature 품질 리포트")
    lines.append("")
    lines.append(f"- 품질 등급: `{grade}`")
    lines.append(f"- rows: {len(sidecar_df):,}")
    lines.append(f"- join 누락: {join_missing:,}")
    lines.append(f"- prev2_missing=1: {int(missing.sum()):,} ({float(missing.mean()):.2%})")
    lines.append(f"- prev2_missing=0: {int(present.sum()):,} ({float(present.mean()):.2%})")
    if raw_stats:
        lines.append(f"- base rows scanned: {raw_stats.get('base_rows'):,}")
        lines.append(f"- collision rows: {raw_stats.get('collision_rows'):,}")
    lines.append("")
    lines.append("## 지적사항")
    failed = [name for name, ok in checks.items() if not ok]
    lines.append("- none" if not failed else "- 실패 checks: `" + "`, `".join(failed) + "`")
    lines.append("")
    lines.append("## 검증 근거 확인")
    for name, ok in checks.items():
        lines.append(f"- {name}: {'pass' if ok else 'fail'}")
    lines.append("")
    lines.append("## prev2_gap_days distribution")
    for q, value in quantiles.items():
        lines.append(f"- p{int(q * 100)}: {value:.0f}")
    if present.any():
        lines.append(f"- max: {prev2_gap[present].max():.0f}")
    lines.append("")
    lines.append("## Yearly missing rate")
    lines.append(md_table(yearly, floatfmt=".6f"))
    lines.append("")
    lines.append("## 검증 공백")
    lines.append("- `prev2`는 기존 `transactions.csv` schema를 바꾸지 않는 sidecar feature입니다.")
    lines.append("- 동일 거래일은 day buffer가 flush된 뒤 history에 들어가므로 현재 거래의 history로 쓰이지 않습니다.")
    lines.append(f"- sidecar_csv: `{FEATURE_PATH}`")
    FEATURE_QUALITY_REPORT_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
    assert grade == "Pass", failed
    return {"grade": grade, "checks": checks, "join_missing": join_missing, "missing_rate": float(missing.mean())}

raw_stats = None
if REBUILD_FEATURES or not FEATURE_PATH.exists():
    raw_stats = build_prev2_feature_file(raw_df)

prev2_df = pd.read_csv(
    FEATURE_PATH,
    dtype={"transaction_id": "string", "prev2_missing": "Int8"},
    parse_dates=["prev2_source_deal_date"],
)
feature_quality = validate_prev2_sidecar(raw_df, prev2_df, raw_stats=raw_stats)
print(feature_quality)
print(FEATURE_PATH)
print(FEATURE_QUALITY_REPORT_PATH)


In [ ]:

# 4) Policy B 필터링, feature 생성, split 구성
model_df = raw_df.merge(prev2_df, on="transaction_id", how="left", validate="one_to_one")
assert int(model_df["prev2_missing"].isna().sum()) == 0
model_df["trade_type"] = model_df["trade_type"].fillna("unknown")
model_df = model_df.loc[(model_df["is_cancelled"] == 0) & (model_df["trade_type"].isin(["중개거래", "unknown"]))].copy()
print("policy_b rows", len(model_df))

NUMERIC_BASE_FEATURES = [
    "area_m2", "floor", "is_basement_floor", "age_years",
    "log_complex_prev_price_per_m2", "complex_prev_missing", "prev_deal_gap_months",
]
NUMERIC_F09_FEATURES = NUMERIC_BASE_FEATURES + [
    "log_complex_prev2_price_per_m2", "prev2_missing", "prev2_gap_months",
]
NUMERIC_F10_FEATURES = NUMERIC_F09_FEATURES + [
    "prev1_prev2_log_return", "prev1_prev2_gap_months",
]
BASE_EMBEDDING_FEATURES = ["legal_dong_code", "sgg_code", "prev_deal_gap_bucket"]
EMBEDDING_DIMS = {
    "legal_dong_code": 16,
    "sgg_code": 8,
    "prev_deal_gap_bucket": 3,
}
HARD_LEAKAGE_COLUMNS = {
    "target", "price_total", "price_per_m2", "deal_date", "transaction_id", "trade_type", "is_cancelled",
    "complex_id", "complex_prev_price_per_m2", "prev_deal_gap_days", "complex_prev2_price_per_m2",
    "prev2_gap_days", "prev2_source_deal_date",
}
for feature_set in [NUMERIC_BASE_FEATURES, NUMERIC_F09_FEATURES, NUMERIC_F10_FEATURES]:
    assert not ((set(feature_set) | set(BASE_EMBEDDING_FEATURES)) & HARD_LEAKAGE_COLUMNS)

GAP_BUCKET_BINS = [-np.inf, 30, 90, 180, 365, np.inf]
GAP_BUCKET_LABELS = ["0-30", "31-90", "91-180", "181-365", "366+"]
GAP_PLUS_BINS = [-np.inf, 0, 7, 14, 30, 60, 90, 180, 365, 730, np.inf]
GAP_PLUS_LABELS = ["missing_or_negative", "1-7", "8-14", "15-30", "31-60", "61-90", "91-180", "181-365", "366-730", "731+"]
AREA_BINS = [-np.inf, 40, 60, 85, 102, 135, np.inf]
AREA_LABELS = ["<=40", "40-60", "60-85", "85-102", "102-135", "135+"]


def gap_bucket(days):
    bucket = pd.Series("missing", index=days.index, dtype="string")
    bucket[(days >= 0) & (days <= 30)] = "0-30"
    bucket[(days >= 31) & (days <= 90)] = "31-90"
    bucket[(days >= 91) & (days <= 180)] = "91-180"
    bucket[(days >= 181) & (days <= 365)] = "181-365"
    bucket[days >= 366] = "366+"
    return bucket.fillna("missing").astype("string")


def gap_plus_bucket(days):
    return pd.cut(days.fillna(-1), bins=GAP_PLUS_BINS, labels=GAP_PLUS_LABELS).astype("string").fillna("missing")


def add_model_features(input_df):
    out = input_df.copy()
    out["target"] = np.log(out["price_per_m2"].astype("float64"))
    out["is_basement_floor"] = (out["floor"] < 0).astype("float32")
    prev1_price = out["complex_prev_price_per_m2"].astype("float64")
    prev2_price = out["complex_prev2_price_per_m2"].astype("float64")
    out["log_complex_prev_price_per_m2"] = np.where(prev1_price > 0, np.log(prev1_price), np.nan)
    out["log_complex_prev2_price_per_m2"] = np.where(prev2_price > 0, np.log(prev2_price), np.nan)
    out["complex_prev_missing"] = out["complex_prev_missing"].fillna(1).astype("float32")
    out["prev2_missing"] = out["prev2_missing"].fillna(1).astype("float32")
    out["prev_deal_gap_months"] = out["prev_deal_gap_days"].astype("float64") / 30.4375
    out["prev2_gap_months"] = out["prev2_gap_days"].astype("float64") / 30.4375
    out["prev1_prev2_gap_months"] = out["prev1_prev2_gap_days"].astype("float64") / 30.4375
    out["prev_deal_gap_bucket"] = gap_bucket(out["prev_deal_gap_days"].astype("float64"))
    out["prev2_gap_bucket_plus"] = gap_plus_bucket(out["prev2_gap_days"].astype("float64"))
    out["prev1_gap_bucket_plus"] = gap_plus_bucket(out["prev_deal_gap_days"].astype("float64"))
    out["prev1_prev2_gap_bucket_plus"] = gap_plus_bucket(out["prev1_prev2_gap_days"].astype("float64"))
    out["area_bucket"] = pd.cut(out["area_m2"], bins=AREA_BINS, labels=AREA_LABELS).astype("string").fillna("missing")
    for feature in ["complex_id", *BASE_EMBEDDING_FEATURES]:
        if feature in out.columns:
            out[feature] = out[feature].fillna("missing").astype("string")
    return out

model_df = add_model_features(model_df)
assert model_df["target"].notna().all()
assert np.isfinite(model_df["target"]).all()


def split_frames(policy_df):
    splits = {
        "train": policy_df.loc[policy_df["deal_date"] <= "2023-12-31"],
        "valid": policy_df.loc[(policy_df["deal_date"] >= "2024-01-01") & (policy_df["deal_date"] <= "2024-12-31")],
        "test": policy_df.loc[(policy_df["deal_date"] >= "2025-01-01") & (policy_df["deal_date"] <= "2025-12-31")],
        "recent_holdout": policy_df.loc[policy_df["deal_date"] >= "2026-01-01"],
    }
    for name, frame in splits.items():
        assert len(frame) > 0, name
    return splits


def apply_smoke_sampling(splits):
    if RUN_MODE != "smoke":
        return {key: value.copy() for key, value in splits.items()}
    out = {}
    for name, frame in splits.items():
        limit = SMOKE_LIMITS[name]
        out[name] = frame.sample(n=limit, random_state=RANDOM_STATE).sort_values("deal_date") if len(frame) > limit else frame.copy()
    return out

full_splits = split_frames(model_df)
run_splits = apply_smoke_sampling(full_splits)
counts_df = pd.DataFrame([{"split": s, "full_rows": len(full_splits[s]), "run_rows": len(run_splits[s])} for s in SPLIT_ORDER])
assert (counts_df["run_rows"] > 0).all()
display(counts_df)


In [ ]:

# 5) 모델 helper
EXPERIMENTS = [
    {
        "experiment_name": "F09_prev2_level",
        "learning_rate": 0.001,
        "loss": "mse",
        "numeric_features": NUMERIC_F09_FEATURES,
        "embedding_features": BASE_EMBEDDING_FEATURES,
        "embedding_dims": EMBEDDING_DIMS,
        "dense_units": [128, 64],
        "seed_offset": 9,
    },
    {
        "experiment_name": "F10_prev2_trend",
        "learning_rate": 0.001,
        "loss": "mse",
        "numeric_features": NUMERIC_F10_FEATURES,
        "embedding_features": BASE_EMBEDDING_FEATURES,
        "embedding_dims": EMBEDDING_DIMS,
        "dense_units": [128, 64],
        "seed_offset": 10,
    },
]


def numeric_features(config):
    return list(config["numeric_features"])


def embedding_features(config):
    return list(config.get("embedding_features", BASE_EMBEDDING_FEATURES))


def numeric_medians_for(config):
    return run_splits["train"][numeric_features(config)].median(numeric_only=True).astype("float32")


def base_log(split_df, medians):
    return split_df["log_complex_prev_price_per_m2"].fillna(medians["log_complex_prev_price_per_m2"]).to_numpy(dtype="float32")


def make_inputs(split_df, config, medians):
    num_features = numeric_features(config)
    numeric_df = split_df[num_features].copy().fillna(medians)
    inputs = {"numeric_input": numeric_df.to_numpy(dtype="float32")}
    for feature in embedding_features(config):
        values = np.asarray(split_df[feature].fillna("missing").astype("string").astype(str).tolist(), dtype=str).reshape(-1, 1)
        inputs[f"{feature}_input"] = tf.convert_to_tensor(values, dtype=tf.string)
    return inputs


def y_for(split_df, medians):
    return split_df["target"].to_numpy(dtype="float32") - base_log(split_df, medians)


def final_log_pred(split_df, raw_pred, medians):
    raw_pred = np.asarray(raw_pred, dtype="float64").reshape(-1)
    return base_log(split_df, medians).astype("float64") + raw_pred


def build_preprocessors(config, train_df, medians):
    features = embedding_features(config)
    train_inputs = make_inputs(train_df, config, medians)
    normalizer = keras.layers.Normalization(name="numeric_normalization")
    normalizer.adapt(train_inputs["numeric_input"])
    lookups = {}
    for feature in features:
        lookup = keras.layers.StringLookup(num_oov_indices=1, mask_token=None, name=f"{feature}_lookup")
        lookup.adapt(train_inputs[f"{feature}_input"])
        lookups[feature] = lookup
    return features, train_inputs, normalizer, lookups


def build_model(config, features, normalizer, lookups):
    tf.keras.utils.set_random_seed(RANDOM_STATE + int(config.get("seed_offset", 0)))
    numeric_input = keras.Input(shape=(len(numeric_features(config)),), name="numeric_input", dtype="float32")
    parts = [normalizer(numeric_input)]
    inputs = [numeric_input]
    for feature in features:
        inp = keras.Input(shape=(1,), name=f"{feature}_input", dtype=tf.string)
        idx = lookups[feature](inp)
        dim = int(config["embedding_dims"].get(feature, EMBEDDING_DIMS[feature]))
        emb = keras.layers.Embedding(lookups[feature].vocabulary_size(), dim, name=f"{feature}_embedding")(idx)
        inputs.append(inp)
        parts.append(keras.layers.Flatten(name=f"{feature}_flatten")(emb))
    x = keras.layers.Concatenate(name="feature_concat")(parts)
    for unit in config["dense_units"]:
        x = keras.layers.Dense(unit, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-5))(x)
        x = keras.layers.Dropout(0.10 if unit >= 128 else 0.05)(x)
    out = keras.layers.Dense(1)(x)
    model = keras.Model(inputs=inputs, outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    return model


def masked_mean(values, mask):
    values = np.asarray(values, dtype="float64")
    mask = np.asarray(mask, dtype=bool)
    if mask.sum() == 0:
        return np.nan
    return float(np.nanmean(values[mask]))


def metric_row(split_df, pred_log, config, split_name):
    y_true = split_df["target"].to_numpy(dtype="float64")
    pred_log = np.asarray(pred_log, dtype="float64").reshape(-1)
    pred_ppm = np.exp(pred_log)
    actual_ppm = split_df["price_per_m2"].to_numpy(dtype="float64")
    pred_total = pred_ppm * split_df["area_m2"].to_numpy(dtype="float64")
    actual_total = split_df["price_total"].to_numpy(dtype="float64")
    abs_log = np.abs(pred_log - y_true)
    abs_pct = np.abs((pred_ppm - actual_ppm) / actual_ppm)
    top_1pct_cutoff = np.quantile(abs_log, 0.99)
    prev2_missing = split_df["prev2_missing"].astype("float64").to_numpy() == 1
    prev2_gap = split_df["prev2_gap_days"].astype("float64").to_numpy()
    prev1_gap = split_df["prev_deal_gap_days"].astype("float64").to_numpy()
    out = {
        "run_mode": RUN_MODE,
        "experiment_name": config["experiment_name"],
        "loss": config.get("loss", "mse"),
        "learning_rate": config["learning_rate"],
        "numeric_features": json.dumps(numeric_features(config), ensure_ascii=False),
        "embedding_features": json.dumps(embedding_features(config), ensure_ascii=False),
        "embedding_dims": json.dumps(config["embedding_dims"], ensure_ascii=False),
        "dense_units": json.dumps(config["dense_units"]),
        "split": split_name,
        "rows": len(split_df),
        "log_mae": float(mean_absolute_error(y_true, pred_log)),
        "log_rmse": float(math.sqrt(mean_squared_error(y_true, pred_log))),
        "price_per_m2_mae": float(mean_absolute_error(actual_ppm, pred_ppm)),
        "price_per_m2_mape": float(np.mean(abs_pct)),
        "total_price_mae_manwon": float(mean_absolute_error(actual_total, pred_total)),
        "abs_pct_error_p50": float(np.quantile(abs_pct, 0.50)),
        "abs_pct_error_p90": float(np.quantile(abs_pct, 0.90)),
        "abs_pct_error_p95": float(np.quantile(abs_pct, 0.95)),
        "abs_pct_error_p99": float(np.quantile(abs_pct, 0.99)),
        "top_1pct_excluded_log_mae": float(np.mean(abs_log[abs_log <= top_1pct_cutoff])),
        "prev1_gap_366plus_log_mae": masked_mean(abs_log, prev1_gap >= 366),
        "prev1_gap_731plus_log_mae": masked_mean(abs_log, prev1_gap >= 731),
        "prev2_missing_log_mae": masked_mean(abs_log, prev2_missing),
        "prev2_gap_366plus_log_mae": masked_mean(abs_log, prev2_gap >= 366),
        "prev2_gap_731plus_log_mae": masked_mean(abs_log, prev2_gap >= 731),
    }
    for threshold in ERROR_RATE_THRESHOLDS:
        key = f"error_gt_{int(threshold * 100)}pct_rate"
        out[key] = float((abs_pct > threshold).mean())
        out[key.replace("rate", "rows")] = int((abs_pct > threshold).sum())
    return out


def prediction_frame(split_df, pred_log, split_name, experiment_name):
    out_cols = [
        "transaction_id", "deal_date", "complex_id", "legal_dong_code", "sgg_code", "area_m2", "floor", "age_years",
        "price_total", "price_per_m2", "target", "complex_prev_missing", "prev_deal_gap_days", "prev_deal_gap_bucket",
        "complex_prev2_price_per_m2", "prev2_missing", "prev2_gap_days", "prev2_gap_bucket_plus",
        "prev1_prev2_log_return", "prev1_prev2_gap_days", "prev1_prev2_gap_bucket_plus", "area_bucket",
    ]
    out = split_df[out_cols].copy()
    out.insert(0, "experiment_name", experiment_name)
    out.insert(1, "split", split_name)
    out["pred_target"] = np.asarray(pred_log, dtype="float64")
    out["log_error"] = out["pred_target"] - out["target"].astype("float64")
    out["abs_log_error"] = out["log_error"].abs()
    out["pred_price_per_m2"] = np.exp(out["pred_target"])
    out["price_per_m2_error"] = out["pred_price_per_m2"] - out["price_per_m2"].astype("float64")
    out["abs_pct_error"] = (out["price_per_m2_error"] / out["price_per_m2"].astype("float64")).abs()
    out["pred_total"] = out["pred_price_per_m2"] * out["area_m2"].astype("float64")
    out["total_error_manwon"] = out["pred_total"] - out["price_total"].astype("float64")
    return out


def train_and_predict(config):
    tf.keras.backend.clear_session()
    print("\n===", config["experiment_name"], "===")
    assert "complex_id" not in embedding_features(config)
    assert "deal_year" not in embedding_features(config)
    assert not ((set(numeric_features(config)) | set(embedding_features(config))) & HARD_LEAKAGE_COLUMNS)
    medians = numeric_medians_for(config)
    features, train_inputs, normalizer, lookups = build_preprocessors(config, run_splits["train"], medians)
    model = build_model(config, features, normalizer, lookups)
    valid_inputs = make_inputs(run_splits["valid"], config, medians)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5, min_lr=1e-5),
    ]
    start = time.perf_counter()
    history = model.fit(
        train_inputs,
        y_for(run_splits["train"], medians),
        validation_data=(valid_inputs, y_for(run_splits["valid"], medians)),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )
    duration = time.perf_counter() - start
    hdf = pd.DataFrame(history.history)
    hdf.insert(0, "epoch", np.arange(1, len(hdf) + 1))
    hdf.insert(0, "experiment_name", config["experiment_name"])
    training = {
        "experiment_name": config["experiment_name"],
        "epochs_ran": len(hdf),
        "best_epoch": int(hdf["val_loss"].idxmin()) + 1,
        "best_val_loss": float(hdf["val_loss"].min()),
        "duration_seconds": duration,
    }
    metrics = []
    pred_frames = {}
    for split_name in SPLIT_ORDER:
        inputs = make_inputs(run_splits[split_name], config, medians)
        raw_pred = model.predict(inputs, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
        pred_log = final_log_pred(run_splits[split_name], raw_pred, medians)
        assert np.isfinite(pred_log).all(), (config["experiment_name"], split_name)
        assert (np.exp(pred_log) > 0).all(), (config["experiment_name"], split_name)
        metrics.append(metric_row(run_splits[split_name], pred_log, config, split_name))
        if split_name in EVAL_SPLITS:
            pred_frames[split_name] = prediction_frame(run_splits[split_name], pred_log, split_name, config["experiment_name"])
    return {"config": config, "history": hdf, "training": training, "metrics": pd.DataFrame(metrics), "pred_frames": pred_frames}


In [ ]:

# 6) F09/F10 실행과 metrics 저장
results = []
metrics_frames = []
prediction_frames = {}
training_rows = []
for config in EXPERIMENTS:
    result = train_and_predict(config)
    results.append(result)
    metrics_frames.append(result["metrics"])
    training_rows.append(result["training"])
    for split_name, frame in result["pred_frames"].items():
        prediction_frames[(config["experiment_name"], split_name)] = frame

metrics_df = pd.concat(metrics_frames, ignore_index=True)
metrics_df["reference_e06_log_mae"] = metrics_df["split"].map(REFERENCE_E06_LOG_MAE)
metrics_df["delta_vs_reference_e06"] = metrics_df["log_mae"] - metrics_df["reference_e06_log_mae"]
metrics_df["beats_reference_e06"] = metrics_df["delta_vs_reference_e06"] < -1e-9
metrics_df.to_csv(METRICS_PATH, index=False)
display(metrics_df.loc[metrics_df["split"].isin(EVAL_SPLITS), [
    "experiment_name", "split", "rows", "log_mae", "delta_vs_reference_e06",
    "abs_pct_error_p95", "abs_pct_error_p99", "error_gt_20pct_rate"
]])
print(METRICS_PATH)


In [ ]:

# 7) group metrics 저장
GROUP_TYPES = [
    "prev2_missing_group",
    "prev2_gap_bucket_plus",
    "prev1_gap_bucket_plus",
    "prev1_prev2_gap_bucket_plus",
    "area_bucket",
]


def prepare_group_frame(pred_df):
    out = pred_df.copy()
    out["prev2_missing_group"] = out["prev2_missing"].astype("Int64").astype("string")
    out["prev1_gap_bucket_plus"] = gap_plus_bucket(out["prev_deal_gap_days"].astype("float64"))
    out["prev2_gap_bucket_plus"] = gap_plus_bucket(out["prev2_gap_days"].astype("float64"))
    out["prev1_prev2_gap_bucket_plus"] = gap_plus_bucket(out["prev1_prev2_gap_days"].astype("float64"))
    return out


def summarize_group(frame, group_type):
    work = frame[[group_type, "abs_log_error", "abs_pct_error", "pred_price_per_m2", "price_per_m2"]].copy()
    work["signed_pct_error"] = (work["pred_price_per_m2"].astype("float64") - work["price_per_m2"].astype("float64")) / work["price_per_m2"].astype("float64")
    grouped = work.groupby(group_type, dropna=False, observed=True)
    out = grouped.agg(
        rows=("abs_log_error", "size"),
        log_mae=("abs_log_error", "mean"),
        price_per_m2_mape=("abs_pct_error", "mean"),
        median_abs_pct_error=("abs_pct_error", "median"),
        p90_abs_pct_error=("abs_pct_error", lambda s: s.quantile(0.90)),
        p95_abs_pct_error=("abs_pct_error", lambda s: s.quantile(0.95)),
        p99_abs_pct_error=("abs_pct_error", lambda s: s.quantile(0.99)),
        error_gt_20pct_rate=("abs_pct_error", lambda s: (s > 0.20).mean()),
        mean_signed_pct_error=("signed_pct_error", "mean"),
    ).reset_index()
    out = out.rename(columns={group_type: "group_value"})
    out.insert(0, "group_type", group_type)
    out["group_value"] = out["group_value"].astype("string")
    return out

group_rows = []
for (experiment_name, split_name), pred_df in prediction_frames.items():
    ready = prepare_group_frame(pred_df)
    for group_type in GROUP_TYPES:
        summary = summarize_group(ready, group_type)
        summary.insert(0, "split", split_name)
        summary.insert(0, "experiment_name", experiment_name)
        group_rows.append(summary)

group_metrics_df = pd.concat(group_rows, ignore_index=True)
group_metrics_df.to_csv(GROUP_METRICS_PATH, index=False)
display(group_metrics_df.head())
print(GROUP_METRICS_PATH)


In [ ]:

# 8) 요약 Markdown 생성
summary_metrics = metrics_df.loc[metrics_df["split"].isin(EVAL_SPLITS)].copy()
summary_metrics["log_mae_pct_equiv"] = (np.exp(summary_metrics["log_mae"]) - 1.0) * 100.0
summary_metrics["p95_pct"] = summary_metrics["abs_pct_error_p95"] * 100.0
summary_metrics["p99_pct"] = summary_metrics["abs_pct_error_p99"] * 100.0
summary_metrics["error_gt_20pct_rate_pct"] = summary_metrics["error_gt_20pct_rate"] * 100.0
summary_metrics["reference_e06_log_mae_pct_equiv"] = (np.exp(summary_metrics["reference_e06_log_mae"]) - 1.0) * 100.0
summary_metrics["log_mae_pct_equiv_delta_vs_e06"] = summary_metrics["log_mae_pct_equiv"] - summary_metrics["reference_e06_log_mae_pct_equiv"]

pivot_log = summary_metrics.pivot(index="experiment_name", columns="split", values="log_mae").reset_index()

percent_error_table = summary_metrics[[
    "experiment_name", "split", "rows", "log_mae_pct_equiv", "log_mae_pct_equiv_delta_vs_e06",
    "p95_pct", "p99_pct", "error_gt_20pct_rate_pct",
    "error_gt_10pct_rate", "error_gt_20pct_rate", "error_gt_30pct_rate", "error_gt_50pct_rate",
]].copy()
for col in ["error_gt_10pct_rate", "error_gt_20pct_rate", "error_gt_30pct_rate", "error_gt_50pct_rate"]:
    percent_error_table[col] = percent_error_table[col] * 100.0
percent_error_table = percent_error_table.rename(columns={
    "log_mae_pct_equiv": "log_mae_percent_equiv",
    "log_mae_pct_equiv_delta_vs_e06": "log_mae_percent_delta_vs_e06",
    "p95_pct": "p95_abs_error_percent",
    "p99_pct": "p99_abs_error_percent",
    "error_gt_20pct_rate_pct": "error_gt_20pct_percent",
    "error_gt_10pct_rate": "error_gt_10pct_percent",
    "error_gt_30pct_rate": "error_gt_30pct_percent",
    "error_gt_50pct_rate": "error_gt_50pct_percent",
})

focus_tail_cols = [
    "experiment_name", "split", "abs_pct_error_p95", "abs_pct_error_p99", "error_gt_20pct_rate",
    "prev2_missing_log_mae", "prev2_gap_366plus_log_mae", "prev2_gap_731plus_log_mae",
]
tail_table = summary_metrics.loc[summary_metrics["split"].isin(EVAL_SPLITS), focus_tail_cols].copy()

focus_groups = group_metrics_df.loc[
    (group_metrics_df["split"].isin(EVAL_SPLITS))
    & (
        ((group_metrics_df["group_type"] == "prev2_missing_group") & (group_metrics_df["group_value"] == "1"))
        | ((group_metrics_df["group_type"] == "prev2_gap_bucket_plus") & (group_metrics_df["group_value"].isin(["366-730", "731+"])))
    )
].copy()

# 판단: F09/F10 후보만 노출하고, 비교 기준은 E06 reference log_mae를 사용한다.
judgement_rows = []
for name in ["F09_prev2_level", "F10_prev2_trend"]:
    cand = summary_metrics.loc[summary_metrics["experiment_name"] == name].set_index("split")
    recent_delta = float(cand.loc["recent_holdout", "delta_vs_reference_e06"])
    test_delta = float(cand.loc["test", "delta_vs_reference_e06"])
    valid_delta = float(cand.loc["valid", "delta_vs_reference_e06"])
    stable_recent = recent_delta <= 0.0005
    stable_test = test_delta <= 0.0005
    clear_recent_gain = recent_delta < -0.0003
    success = stable_recent and stable_test and clear_recent_gain
    severe_degrade = recent_delta > 0.002 or test_delta > 0.002
    judgement = "성공" if success else ("실패" if severe_degrade else "보류")
    judgement_rows.append({
        "experiment_name": name,
        "valid_delta_vs_e06_reference": valid_delta,
        "test_delta_vs_e06_reference": test_delta,
        "recent_delta_vs_e06_reference": recent_delta,
        "judgement": judgement,
    })
judgement_df = pd.DataFrame(judgement_rows)
overall = "성공" if (judgement_df["judgement"] == "성공").any() else ("실패" if (judgement_df["judgement"] == "실패").all() else "보류")

lines = []
lines.append("# E07 prev2 거래 feature 실험 요약")
lines.append("")
lines.append("## 1. 결론")
lines.append(f"- 결론: `{overall}`")
lines.append("- 표에는 `F09`, `F10` 후보만 남겼고, 비교 기준은 `REFERENCE_E06_LOG_MAE`입니다.")
lines.append("- 판단 기준: 평균 log_mae보다 `recent_holdout`, p95/p99, `error_gt_20pct_rate`, prev2 missing/장기 공백 group 안정성을 우선했다.")
lines.append("")
lines.append(md_table(judgement_df))
lines.append("")
lines.append("## 2. 실행 설정")
lines.append(f"- run_mode: `{RUN_MODE}`")
lines.append(f"- max_epochs: `{MAX_EPOCHS}`")
lines.append(f"- batch_size: `{BATCH_SIZE}`")
lines.append("- split: `train<=2023`, `valid=2024`, `test=2025`, `recent_holdout>=2026`")
lines.append("- Policy B: `is_cancelled == 0`, `trade_type in [중개거래, unknown]`")
lines.append("")
lines.append("## 3. Split row 수")
lines.append(md_table(counts_df, floatfmt=".0f"))
lines.append("")
lines.append("## 4. 핵심 log_mae")
lines.append(md_table(pivot_log))
lines.append("")
lines.append("## 5. 퍼센트 오차표")
lines.append(md_table(percent_error_table, floatfmt=".2f"))
lines.append("")
lines.append("## 6. tail 수치")
lines.append(md_table(tail_table))
lines.append("")
lines.append("## 7. missing/366+/731+ group 수치")
if len(focus_groups):
    lines.append(md_table(focus_groups[["experiment_name", "split", "group_type", "group_value", "rows", "log_mae", "p95_abs_pct_error", "p99_abs_pct_error", "error_gt_20pct_rate"]]))
else:
    lines.append("- group rows 없음")
lines.append("")
lines.append("## 8. 다음 판단")
if overall == "성공":
    lines.append("- `prev2`는 후속 slice에서 CSV 생성기/schema 확장 후보로 올릴 수 있다.")
    lines.append("- 뉴스 feature는 `prev2` 반영 방향을 먼저 결정한 뒤 진행한다.")
elif overall == "보류":
    lines.append("- `prev2`를 즉시 `transactions.csv` schema에 반영하지 말고, seed/epoch 반복 또는 outlier 처리 후 재판단한다.")
    lines.append("- 뉴스 feature는 가능하지만, 비교 기준은 E06 reference와 이번 E07 수치를 함께 고정해야 한다.")
else:
    lines.append("- `prev2`는 현재 형태로 CSV 생성기에 반영하지 않는다.")
    lines.append("- 뉴스 feature는 E06 기준선에서 별도 실험으로 진행한다.")
lines.append("")
lines.append("## 9. 생성 산출물")
for path in [FEATURE_PATH, FEATURE_QUALITY_REPORT_PATH, METRICS_PATH, GROUP_METRICS_PATH, SUMMARY_PATH]:
    lines.append(f"- `{path}`")

SUMMARY_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(SUMMARY_PATH)
print("overall", overall)


In [ ]:

# 9) 산출물 생성 smoke check
expected_outputs = [
    FEATURE_PATH,
    FEATURE_QUALITY_REPORT_PATH,
    METRICS_PATH,
    GROUP_METRICS_PATH,
    SUMMARY_PATH,
]
missing_outputs = [str(path) for path in expected_outputs if not path.exists()]
assert not missing_outputs, missing_outputs
assert feature_quality["join_missing"] == 0
assert set(EXPERIMENTS[i]["experiment_name"] for i in range(len(EXPERIMENTS))).issubset(set(metrics_df["experiment_name"].unique()))
assert set(EVAL_SPLITS).issubset(set(metrics_df["split"].unique()))
assert not metrics_df[["log_mae", "abs_pct_error_p95", "abs_pct_error_p99", "error_gt_20pct_rate"]].replace([np.inf, -np.inf], np.nan).isna().any().any()
assert (metrics_df["log_mae"] > 0).all()
print("created", len(expected_outputs), "required outputs")
